# Kafka Producer — Data Engineering Course

This notebook covers two producer examples:
- **Example 1** — Basic producer: send simple messages to a Kafka topic
- **Example 2** — Advanced producer: monitor a live log file and stream its content into Kafka

> Run the **Consumer notebook** in parallel to see messages being consumed in real time.

---
## Example 1 — Basic Producer
Send a couple of hard-coded messages to a Kafka topic.
Great starting point to verify connectivity and understand the core API.

In [ ]:
from kafka import KafkaProducer
from time import sleep

# ----- Configuration -----
TOPIC   = 'kafka-tst-07'
BROKERS = 'course-kafka:9092'

In [ ]:
# Create a KafkaProducer instance.
# By default it connects to the broker and is ready to send bytes.
producer = KafkaProducer(
    bootstrap_servers=BROKERS
)

In [ ]:
# send() is ASYNC — it places the message in an internal buffer and returns immediately.
# The message is NOT guaranteed to be delivered yet at this point.
producer.send(TOPIC, value=b'Hello, World!!!!')

# flush() BLOCKS until all buffered messages have been sent to the broker.
# Always call flush() when you want to guarantee delivery before moving on.
producer.flush()
print('Message 1 sent.')

In [ ]:
# You can also attach a KEY to a message.
# Kafka uses the key to determine which partition the message goes to
# (same key → same partition → guaranteed ordering for that key).
producer.send(TOPIC, key=b'event#2', value=b'This is a Kafka-Python basic tutorial')
producer.flush()
print('Message 2 sent with key.')

---
## Example 2 — File-Monitoring Producer

A more realistic pipeline:
1. A **log generator** (cell below) continuously writes events to a local `.log` file
2. The **producer** tails that file and forwards every new batch of lines to Kafka

Run the log generator first (in a separate thread or notebook), then start the producer.

### Step 1 — Log Generator
Simulates an external system writing events to disk at random intervals.

In [ ]:
from random import randint, random
from time import sleep
from datetime import datetime

LOG_PATH = '/home/developer/kafka/srcFiles/srcFile.log'

def event_generator():
    """Infinite generator that yields (counter, hostname, timestamp) tuples."""
    counter = 1
    while True:
        hostname = f'host{randint(1, 5)}'
        ts = datetime.now().strftime('%a %b %d %H:%M:%S %Y')
        yield counter, hostname, ts
        counter += 1

# Open the log file and write events one by one.
with open(LOG_PATH, 'w') as f:
    for counter, hostname, ts in event_generator():
        line = f'Event #{counter}|{hostname}|{ts}\n'
        f.write(line)
        print(line, end='')

        # flush() here forces the OS to write the buffer to disk immediately,
        # so the producer can see the new line without waiting for the file to close.
        f.flush()

        sleep(5 * random())   # random delay between 0-5 seconds to simulate a real log stream

### Step 2 — Producer (File → Kafka)
Tails the log file and sends each new batch of lines as a single Kafka message.

In [ ]:
from kafka import KafkaProducer
from time import sleep
import json

# ----- Configuration -----
TOPIC2       = 'kafka-tst-02'
BROKERS2     = ['course-kafka:9092']
SOURCE_FILE  = '/home/developer/kafka/srcFiles/srcFile.log'

In [ ]:
# Advanced producer configuration:
producer2 = KafkaProducer(
    bootstrap_servers   = BROKERS2,
    client_id           = 'producer',

    # acks controls the durability guarantee:
    #   0  = fire-and-forget (fastest, no guarantee)
    #   1  = leader ACK only (balanced — used here)
    #  -1  = all in-sync replicas must ACK (slowest, strongest guarantee)
    acks                = 1,

    compression_type    = None,   # Could be 'gzip', 'snappy', 'lz4' to reduce network load

    # retries: how many times to retry a failed send before raising an error
    retries             = 3,

    # Backoff settings control how long to wait between reconnect attempts
    reconnect_backoff_ms     = 50,
    reconnect_backoff_max_ms = 1000
)

In [ ]:
# Tail the log file and forward new lines to Kafka.
with open(SOURCE_FILE, 'r') as f:
    while True:
        lines = f.readlines()   # Returns all lines written since the last read

        if not lines:
            # No new data yet — wait a second and check again (polling pattern)
            sleep(1)
            f.seek(f.tell())    # Seek to current position to "unlock" the file cursor
        else:
            print(f'Sending {len(lines)} line(s): {lines}')

            # Serialize the list of lines to JSON bytes before sending.
            # Kafka messages are raw bytes — you choose the serialization format.
            producer2.send(
                topic = TOPIC2,
                value = json.dumps(lines).encode('utf-8')
            )

            # flush() ensures the message is actually sent to the broker
            # before we sleep and wait for the next batch.
            producer2.flush()

            sleep(3)